In [1]:
from base_agent import BaseAgent
from system_prompts import sys_prompts
import json
import os
from datetime import datetime

In [2]:
llm_name = "deepseek-chat"

In [8]:
def save_json(content, file_path):
    with open(file_path, 'w') as json_file:
        json.dump(content, json_file, indent=4)
        
def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

### ScreenWriter

In [4]:
screenwriter_agent = BaseAgent(llm_type = llm_name, system_prompt=sys_prompts["screenwriterCoT-sys"], use_history=False, temp=0.7)

In [5]:
js = {
    "MovieScript": "春娇和志明 星期天逛公园",
    "Character": [
        "春娇",
        "志明",
    ]
}
movie_script = js["MovieScript"]
characters_list = js["Character"]

In [6]:
first_sentence = movie_script.split(".")[0]
# all_chat.append(query)
previous_sub_script = None
n = 0
index = 1
result = {}

In [ ]:
while True:
    if previous_sub_script: 
        query = f"""
            Script Synopsis: {movie_script}
            Character: {characters_list}
            Previous Sub-Script: {previous_sub_script}
            """
    else:
        query = f"""
            Script Synopsis: {movie_script}
            Character: {characters_list}
            There is no Previous Sub-Script. 
            The current sub-script is the first one. Please start summarizing the first sub-script based on the following content: {first_sentence}.
            """
        
    query = f"""
            Script Synopsis: {movie_script}
            Character: {characters_list}
            """
    
    task_response = screenwriter_agent(query, parse=True)
    # task_response = task_response.replace("'",'"')
    result = task_response
    
    break

save_json(result, "sub_script.json")


In [10]:
import os
import openai

# 禁用所有代理
os.environ['HTTP_PROXY'] = ''
os.environ['HTTPS_PROXY'] = ''
os.environ['NO_PROXY'] = '*'

# 显式指定 OpenAI 客户端不使用代理

In [11]:
data = read_json("sub_script.json")
character_relationships = data['Relationships']
sub_script_list = data['Sub-Script']
sceneplanning_agent = BaseAgent(llm_type = llm_name, system_prompt=sys_prompts["ScenePlanningCoT-sys"], use_history=False, temp=0.7)
data_scene = data
for sub_script_name in sub_script_list:
    sub_script = sub_script_list[sub_script_name]["Plot"]
    query = f"""
                Given the following inputs:
                - Script Synopsis: "{sub_script}"
                - Character Relationships: {character_relationships}
                """
    task_response = sceneplanning_agent(query, parse=True)
    # if "Scene Annotation" not in data_scene[sub_script_name]:
    #     data_scene[sub_script_name]["Scene Annotation"] = []
    
    data_scene['Sub-Script'][sub_script_name]["Scene Annotation"]=task_response
save_json(data_scene, "scene.json")

shotplotcreate_agent = BaseAgent(llm_type = llm_name, system_prompt=sys_prompts["ShotPlotCreateCoT-sys"], use_history=False, temp=0.7)

character_relationships = data_scene['Relationships']
sub_script_list = data_scene['Sub-Script']

```json
{
    "Internal Chain-of-Thought": {
      "Narrative Structure": "The narrative follows a simple, linear structure focusing on a couple's leisurely walk in the park. It's a single-act scene that captures a moment of shared happiness and connection between the characters.",
      "Key Scene Elements": "The key elements include the interaction between 春娇 and 志明, their shared enjoyment of the park, and the moment they admire the flowers together. The scene highlights their relationship dynamics as a couple.",
      "Scene Boundaries": "The scene is self-contained, starting with their decision to walk in the park and ending with them admiring the flowers. There are no major transitions or breaks needed.",
      "Cinematic Elements for Each Scene": "The scene should evoke a sense of warmth and tranquility, with natural lighting and soft colors to match the sunny day. The music should be light and cheerful, complementing the couple's happiness."
    },
    "Scene": {
      "Scene 1"

In [12]:
for sub_script_name in sub_script_list:
    scene_list = sub_script_list[sub_script_name]["Scene Annotation"]["Scene"]
    for scene_name in scene_list:
        scene_details = scene_list[scene_name]
        query = f"""
                    Given the following Scene Details:
                    - Involving Characters: "{scene_details['Involving Characters']}" 
                    - Plot: "{scene_details['Plot']}"
                    - Scene Description: "{scene_details['Scene Description']}"
                    - Emotional Tone: "{scene_details['Emotional Tone']}"
                    - Key Props: {scene_details['Key Props']}
                    - Cinematography Notes: "{scene_details['Cinematography Notes']}"
                    """
                    
        task_response = shotplotcreate_agent(query, parse=True)
        # if "Shot Annotation" not in data_scene[sub_script_name]:
        #     data_scene[sub_script_name]["Shot Annotation"] = []
        
        data_scene['Sub-Script'][sub_script_name]["Scene Annotation"]["Scene"][scene_name]["Shot Annotation"] = task_response
    
        save_json(data_scene, f"{sub_script_name}{scene_name}_Step_3_shot_results.json")

```json
{
    "Internal Chain-of-Thought": {
      "Break Down Scene into Key Shots": "The scene is divided into four key shots: 1) A wide shot of the park to establish the setting, 2) A medium shot of the couple walking hand in hand, 3) A close-up of 春娇 pointing excitedly at the flowers, and 4) A medium close-up of 志明 smiling and following her. Each shot serves to build the warm, happy, and relaxed emotional tone.",
      "Shot Composition and Framing": "Shot 1 is a wide shot to showcase the park's beauty. Shot 2 is a medium shot focusing on the couple's interaction. Shot 3 is a close-up to capture 春娇's excitement. Shot 4 is a medium close-up to highlight 志明's reaction. Framing adheres to the rule of thirds, with characters positioned to lead the viewer's eye naturally.",
      "Character Positioning & Bounding Boxes": "Characters are positioned to avoid overlap, with bounding boxes maximized to focus on their expressions and interactions. 春娇 is on the left, 志明 on the right in Shot 2,